# WS 12.1: Fair Test

In WS 10.2 and 11.2, you trained decision trees and measured how well they did — accuracy, confusion matrix, precision, recall. But every time, you trained the model on all the data and then tested it on *the same data*.

That's like studying the answer key the night before an exam and then taking the same exam. Of course you'll score well — but did you actually learn anything?

Today you'll give your model a **fair test**: train it on one portion of the data, then test it on data it has never seen. Along the way, we'll gain more practice in using precision and recall.

> **I will not use AI tools on this worksheet.**
>
> **Name:** \_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_

### Setup

Run the cell below to load the libraries.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split

---

## Part 1: A Fair Test

### The data

We'll work with a dataset of 30,000 credit card customers from a bank in Taiwan ([source: UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients)). Each row is one customer. The column `default` is **1** if the customer failed to pay their next month's bill (a **default**) and **0** if they paid.

Run the cell below to load the data.

In [ ]:
credit = pd.read_csv("https://raw.githubusercontent.com/statisfactions/QRAI-materials/main/data/taiwan_credit.csv")
credit.head(10)

The dataset has many columns, but we'll focus on four features:

| Feature | What it means |
|---------|-------------|
| `LIMIT_BAL` | The customer's credit limit (in NT dollars) |
| `AGE` | The customer's age |
| `PAY_AMT1` | How much the customer paid on their most recent bill |
| `PAY_0` | How late the customer was on their most recent payment (0 = on time, 1 = one month late, 2 = two months late, etc.) |

**Exercise 1.1:** How many customers are in the dataset? How many defaulted? What fraction of customers defaulted?

> **Hint:** `credit["default"].sum()` counts the defaults. `len(credit)` gives the total.

In [ ]:
# Your code here

*Your answer here.*

**Exercise 1.2:** Now try running `credit["default"].mean()`. What number do you get? Compare it to the fraction you just calculated. Why would `.mean()` of a column of 0s and 1s give the same result?

In [ ]:
# Your code here

*Your answer here.*

**Exercise 1.3:** If we predicted "no default" for every customer — without looking at any features — what would the accuracy be? Does high accuracy mean the model is useful?

*Your answer here.*

### Splitting the data

Here is something worth sitting with: how do we know the model has *actually "learned"* something useful?  We want the model to be able to perform well on *new* data, not just the data we originally fed into the model.

To test the model's ability to perform well on new data, we **split** the data before training. We randomly set aside a portion the model never sees during training and use it as the test.

In [ ]:
features = ["LIMIT_BAL", "AGE", "PAY_AMT1", "PAY_0"]
X = credit[features]
y = credit["default"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

`train_test_split` randomly shuffles the data and splits it into two pieces:
- **Training set** (`X_train`, `y_train`): the model learns from this. Think of it as the **practice problems**.
- **Test set** (`X_test`, `y_test`): we evaluate on this. Think of it as the **real exam**.

`test_size=0.2` means 20% goes to the test set, 80% to training. Because the shuffle is random, everyone in the class gets a slightly different split — and that is fine.

**Exercise 1.4:** How many customers are in your training set? How many in your test set? How many of your test-set customers defaulted?

In [ ]:
# Your code here

*Your answer here.*

**Exercise 1.5:** Compare your test-set default count with a neighbor. Did you get the same number? Explain.

*Your answer here.*

### Training and testing

Now train a decision tree on the training set only. The model will never see the test set during training.

> **Reminder — the decision tree pattern from WS 10.2:**
>
> ```python
> model = DecisionTreeClassifier(max_depth=_____)
> model.fit(_____, _____)
> ```
>
> `max_depth` controls how many questions the tree can ask. `.fit()` takes the features first, then the labels.

**Exercise 1.6:** Create a `DecisionTreeClassifier` with `max_depth=3`. Train it on the **training set** only.

In [ ]:
# Your code here

**Exercise 1.7:** Compute predictions on the **test set** and build the confusion matrix.

> **Reminder — the prediction and confusion matrix pattern from WS 10.2:**
>
> ```python
> predictions = model.predict(_____)
> pd.crosstab(_____, predictions, rownames=["actual"], colnames=["predicted"])
> ```
>
> Fill in the blanks: what data should the model predict on? What are the actual labels to compare against?

In [ ]:
# Your code here

**Exercise 1.8:** Write the four values from your confusion matrix.

> **Reminder:** `pd.crosstab` lists rows and columns in order 0, 1 — so the table looks like this:
>
> |  | Predicted **no default (0)** | Predicted **default (1)** |
> |---|---|---|
> | **Actually paid (0)** | True Negative (TN) | False Positive (FP) |
> | **Actually defaulted (1)** | False Negative (FN) | True Positive (TP) |

TP = \_\_\_ &nbsp;&nbsp; FP = \_\_\_ &nbsp;&nbsp; FN = \_\_\_ &nbsp;&nbsp; TN = \_\_\_

*Your answer here.*

### Seeing the confusion matrix

Like in today's opening activity, we can represent the confusion matrix as a **grid of icons** — each icon stands for a group of customers. **Color** shows the actual outcome: red = actually defaulted, blue = actually paid. **Shape** shows the model's prediction: x = predicted default, o = predicted no default.

Run the cell below to draw the icon array from your confusion matrix. Each icon represents 50 customers.

In [ ]:
#@title Icon array — run this cell
total = len(y_test)
cm = pd.crosstab(y_test, predictions)
TN_count = int(cm.iloc[0, 0])
FP_count = int(cm.iloc[0, 1])
FN_count = int(cm.iloc[1, 0])
TP_count = int(cm.iloc[1, 1])

print(f"TP = {TP_count}   FP = {FP_count}   FN = {FN_count}   TN = {TN_count}")

people_per_icon = 50
n_TN = round(TN_count / people_per_icon)
n_FP = round(FP_count / people_per_icon)
n_FN = round(FN_count / people_per_icon)
n_TP = round(TP_count / people_per_icon)
n_total = n_TN + n_FP + n_FN + n_TP

icons = (
    [("steelblue", "o")] * n_TN +
    [("steelblue", "X")] * n_FP +
    [("firebrick", "o")] * n_FN +
    [("firebrick", "X")] * n_TP
)

cols = 10
n_rows = (n_total + cols - 1) // cols
plt.figure(figsize=(6, 0.6 * n_rows + 2))
for i, (c, m) in enumerate(icons):
    plt.scatter(i % cols, n_rows - 1 - i // cols, c=c, marker=m, s=150,
                edgecolors="black", linewidths=0.5)
plt.xlim(-0.5, cols - 0.5)
plt.ylim(-0.5, n_rows - 0.5)
plt.xticks([])
plt.yticks([])
plt.title(f"Each icon = {people_per_icon} customers")
plt.scatter([], [], c="firebrick", marker="X", s=80, edgecolors="black", linewidths=0.5, label=f"Predicted AND actually defaulted (TP = {TP_count})")
plt.scatter([], [], c="firebrick", marker="o", s=80, edgecolors="black", linewidths=0.5, label=f"Predicted paid but actually defaulted (FN = {FN_count})")
plt.scatter([], [], c="steelblue", marker="X", s=80, edgecolors="black", linewidths=0.5, label=f"Predicted default but actually paid (FP = {FP_count})")
plt.scatter([], [], c="steelblue", marker="o", s=80, edgecolors="black", linewidths=0.5, label=f"Predicted paid AND actually paid (TN = {TN_count})")
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=1, fontsize=9)
plt.tight_layout()
plt.show()

**Exercise 1.9:** Look at the icon array. The x icons are customers the model **flagged for default**. The o icons are customers the model **predicted would pay**.

- Compare the number of red x's to red o's. Is the model catching most of the defaulters, or missing most of them?
- Now compare the blue x's to the red x's. When the model flags someone for default, does it seem to be right more often than wrong, or the other way around?

*Your answer here.*

### Precision — when the model says "default," how often is it right?

The next cell highlights just the customers the model **predicted would default** — the x icons. Everything else is faded.

In [ ]:
#@title Precision view — run this cell
plt.figure(figsize=(6, 0.6 * n_rows + 2))
for i, (c, m) in enumerate(icons):
    x, y = i % cols, n_rows - 1 - i // cols
    if m == "X":
        plt.scatter(x, y, c=c, marker="X", s=150, edgecolors="black", linewidths=0.5)
    else:
        plt.scatter(x, y, c=c, marker="o", s=80, edgecolors="lightgray", linewidths=0.3, alpha=0.15)
plt.xlim(-0.5, cols - 0.5)
plt.ylim(-0.5, n_rows - 0.5)
plt.xticks([])
plt.yticks([])
plt.title("Precision: of the x icons (flagged for default), how many are red?")
plt.scatter([], [], c="firebrick", marker="X", s=80, edgecolors="black", linewidths=0.5, label="Predicted AND actually defaulted (TP)")
plt.scatter([], [], c="steelblue", marker="X", s=80, edgecolors="black", linewidths=0.5, label="Predicted default but actually paid (FP)")
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=2, fontsize=9)
plt.tight_layout()
plt.show()

**Exercise 1.10:** Look at the highlighted x icons — these are everyone the model flagged for default. Some are red (actually defaulted) and some are blue (actually paid).

Just from looking at the picture, roughly what fraction of the x icons are red? This is your visual estimate of **precision** — when the model says "default," how often is it right?

*Your answer here.*

**Exercise 1.11:** Now compute precision exactly. **Precision** = TP / (TP + FP) — the fraction of positive predictions that were correct.

- In plain English: "Of the \_\_\_ customers the model predicted would default, \_\_\_ actually did."

In [ ]:
# Your confusion matrix values (from earlier):
print(f"TP = {TP_count}   FP = {FP_count}   FN = {FN_count}   TN = {TN_count}")

# Compute precision:
# Your code here

*Your answer here.*

### Recall — of everyone who actually defaulted, how many did the model find?

Now the cell highlights just the customers who **actually defaulted** — the red icons. Everything else is faded.

In [ ]:
#@title Recall view — run this cell
plt.figure(figsize=(6, 0.6 * n_rows + 2))
for i, (c, m) in enumerate(icons):
    x, y = i % cols, n_rows - 1 - i // cols
    if c == "firebrick":
        plt.scatter(x, y, c=c, marker=m, s=150, edgecolors="black", linewidths=0.5)
    else:
        plt.scatter(x, y, c=c, marker=m, s=80, edgecolors="lightgray", linewidths=0.3, alpha=0.15)
plt.xlim(-0.5, cols - 0.5)
plt.ylim(-0.5, n_rows - 0.5)
plt.xticks([])
plt.yticks([])
plt.title("Recall: of all red icons (defaulted), how many did the model find (x)?")
plt.scatter([], [], c="firebrick", marker="X", s=80, edgecolors="black", linewidths=0.5, label="Predicted AND actually defaulted (TP)")
plt.scatter([], [], c="firebrick", marker="o", s=80, edgecolors="black", linewidths=0.5, label="Predicted paid but actually defaulted (FN)")
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=2, fontsize=9)
plt.tight_layout()
plt.show()

**Exercise 1.12:** Look at the highlighted red icons — these are everyone who actually defaulted. Some are x's (the model caught them) and some are o's (the model missed them).

Just from looking at the picture, roughly what fraction of the red icons are x's? This is your visual estimate of **recall** — of everyone who actually defaulted, how many did the model find?

*Your answer here.*

**Exercise 1.13:** Now compute recall exactly. **Recall** = TP / (TP + FN) — the fraction of actual positives the model caught.

- In plain English: "Of the \_\_\_ customers who actually defaulted, the model found \_\_\_."

In [ ]:
# Your confusion matrix values (from earlier):
print(f"TP = {TP_count}   FP = {FP_count}   FN = {FN_count}   TN = {TN_count}")

# Compute recall:
# Your code here

*Your answer here.*

**Exercise 1.14:** The bank uses this model to decide who gets a loan.

- A **false positive** means the model says a customer will default, but they would have paid. What might the bank do to this person?
- A **false negative** means the model says a customer will pay, but they actually default. What is the cost to the bank?

If you could improve only one — precision or recall — which would you choose, and why?

*Your answer here.*

---

## Part 2: Deeper Trees, Bigger Gap

In Part 1 you used `max_depth=3`. What happens if we let the tree ask more questions?

### Repeating with a loop

To try many depths, we need to repeat the same steps — train a tree, compute accuracy — for each depth. We could copy-paste the code five times and change the number each time, but there is a better way: a **`for` loop**.

A `for` loop tells Python: "take each item from this list, one at a time, and run the same code on it." Here is the pattern:

```python
for _____ in some_list:
    # code that runs once per item
```

Two things to notice:
- The line with `for` ends with a **colon** (`:`). This tells Python that the indented code below is the body of the loop.
- Everything **indented** below the `for` line runs once for each item.

If you forget the colon, Python will give a `SyntaxError`. If you forget to indent, the code will run only once instead of repeating.

Here is a concrete example:

```python
animals = ["cat", "dog", "fish"]
for animal in animals:
    print(animal)
```

This prints `cat`, then `dog`, then `fish`. The variable name `animal` is your choice — you could call it `x`, `item`, or even `ethanisthebestprofessor` and the code would work the same way. Python doesn't care what you name it. We pick `animal` because it makes the code easier to read when the list contains animals.

**Exercise 2.1:** The list below contains five tree depths. Write a `for` loop that prints each depth.

In [ ]:
depths = [1, 3, 5, 10, 20]
# Your code here

### Trying every depth

We can also use a loop to **collect results**. The idea: before the loop, create an empty list — that's `[]`, a list with nothing in it yet. Then inside the loop, use `.append()` to add one result each time through:

```python
numbers = [1, 2, 3]
doubled = []                  # start with an empty list
for n in numbers:
    doubled.append(n * 2)     # add one result each time
print(doubled)  # [2, 4, 6]
```

Before the loop, `doubled` is `[]`. After the first pass it's `[2]`, then `[2, 4]`, then `[2, 4, 6]`. The list grows by one item each time through the loop.

The code below uses this pattern. For each depth, it trains a tree and computes accuracy, precision, and recall on both the training set and the test set. Run it.

In [ ]:
depths = [1, 3, 5, 10, 20]

# Empty lists to collect results
train_accuracies = []
test_accuracies = []
train_precisions = []
test_precisions = []
train_recalls = []
test_recalls = []

for depth in depths:
    # Train a tree at this depth
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)

    # Accuracy on training set vs. test set
    train_acc = (tree.predict(X_train) == y_train).sum() / len(y_train)
    test_acc = (tree.predict(X_test) == y_test).sum() / len(y_test)
    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)

    # Confusion matrix on training set → precision and recall
    train_pred = tree.predict(X_train)
    train_cm = pd.crosstab(y_train, train_pred)
    tp_tr = int(train_cm.iloc[1, 1])
    fp_tr = int(train_cm.iloc[0, 1])
    fn_tr = int(train_cm.iloc[1, 0])
    train_precisions.append(tp_tr / (tp_tr + fp_tr) if (tp_tr + fp_tr) > 0 else 0)
    train_recalls.append(tp_tr / (tp_tr + fn_tr) if (tp_tr + fn_tr) > 0 else 0)

    # Confusion matrix on test set → precision and recall
    test_pred = tree.predict(X_test)
    test_cm = pd.crosstab(y_test, test_pred)
    tp_te = int(test_cm.iloc[1, 1])
    fp_te = int(test_cm.iloc[0, 1])
    fn_te = int(test_cm.iloc[1, 0])
    test_precisions.append(tp_te / (tp_te + fp_te) if (tp_te + fp_te) > 0 else 0)
    test_recalls.append(tp_te / (tp_te + fn_te) if (tp_te + fn_te) > 0 else 0)

    print(f"Depth {depth:2d}:  train acc = {train_acc:.4f}   test acc = {test_acc:.4f}   train prec = {train_precisions[-1]:.4f}   test prec = {test_precisions[-1]:.4f}   train rec = {train_recalls[-1]:.4f}   test rec = {test_recalls[-1]:.4f}")

**Exercise 2.2:** Plot training accuracy and test accuracy on the same graph.

To put two lines on the same plot, call `plt.plot()` twice before `plt.show()`. Give each line a `label` so the legend can tell them apart:

```python
plt.plot(x_values, y_values_1, marker="o", label="Line 1")
plt.plot(x_values, y_values_2, marker="s", label="Line 2")
plt.xlabel("...")
plt.ylabel("...")
plt.title("...")
plt.legend()
plt.show()
```

Use `depths` as the x-axis and `train_accuracies` / `test_accuracies` as the two y-axes.

In [ ]:
# Your code here

**Exercise 2.3:** Look at the accuracy plot.

- What happens to training accuracy as the tree gets deeper? What happens to test accuracy?
- At around what depth does the gap between training and test accuracy start growing?

*Your answer here.*

A model that performs much better on training data than on new data is called **overfitting**. The model has memorized the training set — including its noise and quirks — instead of learning the real pattern.

**Exercise 2.4:** How is overfitting similar to reward hacking, which we studied in unit 1?

*Your answer here.*

### What about precision and recall?

Accuracy tells us the overall error rate, but you've seen that accuracy can be misleading when one class is much more common. Let's see what happens to precision and recall as the tree gets deeper.

**Exercise 2.5:** Plot training precision and test precision on the same graph, across depths. (Same idea as the accuracy plot — two lines, one for train and one for test.)

In [ ]:
# Your code here

**Exercise 2.6:** In a separate graph, plot training recall and test recall across depths.

In [ ]:
# Your code here

**Exercise 2.7:** Look at the precision and recall plots.

- Remember: precision measures how often the model is right when it flags someone for default. What happens to precision as trees get deeper? What does that mean for the bank's customers?
- Recall measures how many actual defaulters the model catches. Does deeper = better for recall?

*Your answer here.*

**Exercise 2.8:** Based on everything you've seen today, would you recommend the bank use the depth-3 tree or the depth-20 tree? What information would you want before deciding?

*Your answer here.*

---

*Worksheet created by Ethan C. Brown in collaboration with Claude Code.*